# Stage1 现金拖累诊断与回收叠加

诊断生产策略的高现金来自哪里，并测试把部分闲置现金转入 SOXX 的有界叠加是否改善风险调整超额。
使用真实生产目标历史 + qlib 原生价格。


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'stage1_cash_drag_overlay_20261010'
plan_bytes=(CARD/'plan.json').read_bytes(); plan=json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('source:', plan['source_runtime'])


plan sha256: 28f7c08807a8e8f79e7078c8501ac2386d16c806026e68ccdc2d6f2d966b642b
source: runtime/soxl/stage1_futu_sim/execution_state_v2.json


In [2]:
from src.workflow import cash_drag_overlay as workflow
from src.workflow.cash_drag_overlay import write_card_files
summary = workflow.run_cash_drag_overlay(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'], '| best:', summary['best_overlay'], '| improvement:', summary['improvement'])
print('diagnostic:', json.dumps(summary['diagnostic'], ensure_ascii=False))
print('recorder:', summary['recorder_id'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[53655:MainThread](2026-10-09 19:50:31,127) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/09 19:50:31 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[53655:MainThread](2026-10-09 19:50:31,668) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[53655:MainThread](2026-10-09 19:50:31,668) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


...

[53655:MainThread](2026-10-09 19:50:31,774) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_cash_drag_overlay_20261010.


[53655:MainThread](2026-10-09 19:50:31,776) INFO - qlib.workflow - [exp.py:258] - Experiment 528322389147800182 starts running ...


[53655:MainThread](2026-10-09 19:50:31,865) INFO - qlib.workflow - [recorder.py:345] - Recorder 96a92d05788d49148a26e4dd943e9d16 starts running under Experiment 528322389147800182 ...


[53655:MainThread](2026-10-09 19:50:31,963) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


decision: cash_overlay_no_improvement | best: recycle_0 | improvement: False
diagnostic: {"average_raw_beta": 0.5377675253166092, "average_beta_cap": 0.5408710782578476, "average_soxl_weight": 0.16484730807076373, "average_soxx_weight": 0.32914950536322635, "average_cash_weight": 0.5060031865660098, "sessions": 12, "cap_binding_fraction": 0.0, "interpretation": "If cap_binding_fraction is near zero, the cash is the policy's own raw target rather than a hard-cap artifact; the fix is an overlay, not a cap change."}
recorder: 96a92d05788d49148a26e4dd943e9d16


In [3]:
import pandas as pd
display(pd.read_csv(CARD/'overlay_comparison.csv')[[
  'overlay','cost_bps','strategy_return_pct','benchmark_return_pct','strategy_sharpe',
  'benchmark_sharpe','strategy_max_drawdown_pct','average_cash_weight','beats_sharpe']].round(3))


,overlay,cost_bps,strategy_return_pct,benchmark_return_pct,strategy_sharpe,benchmark_sharpe,strategy_max_drawdown_pct,average_cash_weight,beats_sharpe
0,recycle_0,10.0,2.339,0.489,1.006,0.439,-2.897,0.506,True
1,recycle_25,10.0,2.085,0.489,0.919,0.439,-3.090,0.380,True
2,recycle_50,10.0,1.816,0.489,0.838,0.439,-3.283,0.253,True
3,recycle_75,10.0,1.520,0.489,0.758,0.439,-3.476,0.127,True
4,recycle_100,10.0,1.184,0.489,0.677,0.439,-4.009,0.000,True
5,recycle_0,25.0,1.673,0.489,0.812,0.439,-2.904,0.506,True
6,recycle_25,25.0,1.492,0.489,0.755,0.439,-3.097,0.380,True
7,recycle_50,25.0,1.284,0.489,0.698,0.439,-3.290,0.253,True
8,recycle_75,25.0,1.020,0.489,0.633,0.439,-3.483,0.127,True
9,recycle_100,25.0,0.666,0.489,0.554,0.439,-4.031,0.000,True


## 中文结论


In [4]:
print('诊断:', summary['diagnostic']['interpretation'])
print('平均权重 SOXL %.3f SOXX %.3f cash %.3f'%(summary['diagnostic']['average_soxl_weight'],summary['diagnostic']['average_soxx_weight'],summary['diagnostic']['average_cash_weight']))
print('结论:', summary['decision'])


诊断: If cap_binding_fraction is near zero, the cash is the policy's own raw target rather than a hard-cap artifact; the fix is an overlay, not a cap change.
平均权重 SOXL 0.165 SOXX 0.329 cash 0.506
结论: cash_overlay_no_improvement
